# Intracranial Hemorrhage Detection — Training Notebook (Colab / GPU)

Loads the Brain CT Hemorrhage dataset via `kagglehub`, preprocesses images, and trains a CNN binary classifier (Normal vs Hemorrhage).

In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("abdulkader90/brain-ct-hemorrhage-dataset")
print("Path to dataset files:", path)

In [ ]:
import os, glob

data_path = os.path.join(path, "Data")

hemorrhage_images = glob.glob(os.path.join(data_path, "Hemorrhagic", "**", "*.[jJ][pP][gG]"), recursive=True)
normal_images = glob.glob(os.path.join(data_path, "NORMAL", "**", "*.[jJ][pP][gG]"), recursive=True)

print("Hemorrhage images found:", len(hemorrhage_images))
print("Normal images found:", len(normal_images))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for i in range(3):
    img = Image.open(hemorrhage_images[i])
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title("Hemorrhage")
    axes[0, i].axis('off')

    img = Image.open(normal_images[i])
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title("Normal")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import cv2

IMG_SIZE = 224  # standard size for CNNs like ResNet

def load_and_preprocess(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # load as grayscale
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))         # resize to consistent size
    img = img / 255.0                                    # normalize pixel values to 0-1
    return img

# Test it on one image
sample = load_and_preprocess(hemorrhage_images[0])
print("Shape:", sample.shape)
print("Min/Max pixel values:", sample.min(), sample.max())

plt.imshow(sample, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
import numpy as np

X = []  # will hold all processed images
y = []  # will hold labels (0 = normal, 1 = hemorrhage)

for img_path in hemorrhage_images:
    img = load_and_preprocess(img_path)
    X.append(img)
    y.append(1)

for img_path in normal_images:
    img = load_and_preprocess(img_path)
    X.append(img)
    y.append(0)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_r = X_train.reshape(-1, 224, 224, 1)
X_test_r = X_test.reshape(-1, 224, 224, 1)

print("Train shape:", X_train_r.shape)
print("Test shape:", X_test_r.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(224, 224, 1)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(
    X_train_r, y_train,
    validation_data=(X_test_r, y_test),
    epochs=5,
    batch_size=32
)

In [ ]:
model.save('hemorrhage_cnn_model.keras')